# Hands-on I: Empirical Exploration of Theory
#### UAI Tutorial: Scalable Sampling of Bayesian Neural Networks

**Time:** 15 Minutes **Goal:** Interactively explore why "Hybrid" sampling (Optimization + Ensembling + Sampling) is crucial for Bayesian neural networks.

In this short session, we will look at a toy 2D neural network to visualize the weight space and explore key challenges preventing standard MCMC from scaling:

1. How do neural network posterior symmetries look like? 

2. Initialization Efficiency: Getting to the typical set - fast.

3. Multimodality: How to avoid getting stuck in a single mode?

## Setup

First we will provide some setup code and helper functions, you do not need to take a look at every single detail but have a glance at the following functions:

- `network_fn`: Our simple 2D neural network.
- `unnormalized_log_posterior`: The log-posterior function.
- `simulate_data`: Our data generating process.

Notably we use the great `blackjax` library for NUTS sampling. Check out the sampling function to see how easy it is to use! If you want to learn more about `blackjax`, check out their [documentation](https://blackjax-devs.github.io/blackjax/) and the [Sampling Book Project](https://blackjax-devs.github.io/sampling-book/).

In [ ]:
# SETUP

# If you are running this in Google Colab, uncomment the following line to install dependencies
# !pip install -q blackjax jax jaxlib numpy pandas plotnine

# If you work locally either install via pip or see 
# https://github.com/muniq-ai/uai26_sampling_tut_website 
# for setup instructions using uv. 

import jax
import jax.numpy as jnp
import numpy as np
import pandas as pd
import plotnine as p9
import blackjax
import warnings

warnings.filterwarnings('ignore')

# Model & DGP
def network_fn(x, w):
    """
    A simple linear network with a symmetry: f(x) = (w1 * w2) * x.
    Note that w = (1, 1) gives the same output as w = (-1, -1).
    """
    return jnp.prod(w) * x

def unnormalized_log_posterior(w, x, y, prior_mean=0.0, prior_std=1.0, likelihood_std=0.2):
    """Computes log(Prior) + log(Likelihood)"""
    log_prior = jax.scipy.stats.norm.logpdf(w, prior_mean, prior_std).sum()
    log_likelihood = jax.scipy.stats.norm.logpdf(y, network_fn(x, w), likelihood_std).sum()
    return log_prior + log_likelihood

def simulate_data(n=100, key=0):
    """Generates synthetic data from the true parameter w=(1,1)"""
    w_true = jnp.array([1.0, 1.0])
    x = jnp.linspace(-2, 2, n)
    y = network_fn(x, w_true) + jax.random.normal(jax.random.PRNGKey(key), x.shape) * 0.2
    return x, y

# Inference
def optimize_model(y, x, key=0, steps=200, step_size=1e-3, init_w=None):
    """Runs Gradient Descent to find a mode."""
    if init_w is None:
        w = jax.random.normal(jax.random.PRNGKey(key), (2,))
    else:
        w = init_w
    w_init = w.copy()

    @jax.jit
    def loss_fn(w):
        return -unnormalized_log_posterior(w, x, y)

    def update(w, _):
        # Minimize negative log posterior
        grads = jax.grad(loss_fn)(w)
        return w - step_size * grads, None

    w_opt, _ = jax.lax.scan(update, w, None, length=steps)
    return w_opt, w_init


def run_nuts_sampler(y, x, w_init, num_samples=1000, num_warmup=500, seed=0, log_prob_fn=None):
    """Runs a NUTS sampler using BlackJax."""
    rng_key = jax.random.PRNGKey(seed)

    if log_prob_fn is None:
        def logdensity(w): return unnormalized_log_posterior(w, x, y)
    else:
        logdensity = log_prob_fn

    # Adaptation/Warmup Phase
    warmup = blackjax.window_adaptation(blackjax.nuts, logdensity)
    (state, params), _ = warmup.run(rng_key, w_init, num_steps=num_warmup)

    # Sampling Phase
    kernel = blackjax.nuts(logdensity, **params).step
    def one_step(state, key):
        state, _ = kernel(key, state)
        return state, state.position

    keys = jax.random.split(rng_key, num_samples)
    _, samples = jax.lax.scan(one_step, state, keys)
    return samples

# Utils
def plot_weight_space(samples=None, title="Posterior Samples", pointalpha=0.5):
    """Visualizes the 2D weight space and posterior samples."""
    # Generate background hyperbola (Symmetry manifold where w1*w2 = 1)
    t = jnp.linspace(0.2, 3.0, 100)
    hyp_pos = pd.DataFrame({'w1': t, 'w2': 1/t})
    hyp_neg = pd.DataFrame({'w1': -t, 'w2': -1/t})

    base_plot = (p9.ggplot()
                 + p9.geom_line(p9.aes(x='w1', y='w2'), data=hyp_pos, color='red', alpha=0.3, size=1)
                 + p9.geom_line(p9.aes(x='w1', y='w2'), data=hyp_neg, color='red', alpha=0.3, size=1)
                 + p9.xlim(-3, 3) + p9.ylim(-3, 3)
                 + p9.theme_minimal()
                 + p9.labs(x="Weight 1", y="Weight 2", title=title)
                 + p9.theme(figure_size=(5, 5))
                 )

    if samples is not None:
        df = pd.DataFrame(samples, columns=['w1', 'w2'])
        df['step'] = np.arange(len(df))
        base_plot += p9.geom_point(p9.aes(x='w1', y='w2', color='step'), data=df, alpha=pointalpha, size=1.5)
        base_plot += p9.scale_color_gradient(low="blue", high="#aaddff")

    return base_plot

print("Environment Ready. Data Simulated.")
x_data, y_data = simulate_data()

## The Landscape: Symmetries and Initialization

We are training a network $f(x) = (w_1 \cdot w_2) \cdot x$. Even this simple network has **symmetries**. E.g. if we swap signs $(w_1, w_2) \to (-w_1, -w_2)$, the output is identical, but also rescaling can be applied. This creates a multimodal posterior (two "valleys" of high probability).

In the plot below, the Red Lines represent the equal likelihood symmetry manifolds (Hyperbolas $w_1 w_2 = 1$).

In [ ]:
plot_weight_space(title="Weight Space with Exact Symmetry Manifold")

### Exercise 1: Random vs. Optimized Initialization

Standard MCMC theory assumes we converge "eventually." But in realistic deep learning, "eventually" takes too long.

Task: Compare the two plots below.

- **Case 1 (Random):** Initialize NUTS from a random standard Gaussian draw.
- **Case 2 (Hybrid):** Run a quick Optimization (Gradient Descent) first, then start NUTS.

Try changing `SEED` to see how optimization stabilizes sampling.

In [ ]:
# (Initial) Configuration
# Intentionally short to highlight initialization issues even on this simple problem
N_SAMPLES = 16
N_WARMUP = 8   
SEED = 0 # change

# 1. Random Init
print("Running Sampler 1 (Random Init)...")
w_random = jax.random.normal(jax.random.PRNGKey(SEED), (2,)) * 0.01
samples_rand = run_nuts_sampler(y_data, x_data, w_random, N_SAMPLES, N_WARMUP, seed=SEED)
plot1 = plot_weight_space(samples_rand, title=f"Case 1: Random Init (SEED={SEED})")
plot1

In [ ]:
# 2. Optimized Init
SEED = 0 # change - sampling should now reliably find the symmetric manifold/mode
print("Running Sampler 2 (Optimized Init)...")
w_opt, _ = optimize_model(y_data, x_data, key=SEED, steps=100, step_size=0.0001, init_w=w_random)
print(f"Optimized Init Found: w_opt = {w_opt}")
samples_opt = run_nuts_sampler(y_data, x_data, w_opt, N_SAMPLES, N_WARMUP, seed=SEED)
plot2 = plot_weight_space(samples_opt, title=f"Case 2: Optimized Init (SEED={SEED})")
plot2

#### Discussion

- Case 1: Often you see a blue trail moving towards the red line or sometimes not even reaching it. This is the potentially wasteful burn-in phase.

- Case 2: The sampler starts on the manifold. We waste zero samples finding a high-likelihood region.

But at what cost? Optimization is not free. Analyze the time taken for both cases. How heavy is the bill for optimization?

In [ ]:
# Timing Analysis (Optional: implement replications for more robust estimates)
import time

# first time the sampling:
start_time = time.time()
samples_opt = run_nuts_sampler(y_data, x_data, w_opt, N_SAMPLES, N_WARMUP, seed=SEED)
end_time = time.time()
time_sampling = end_time - start_time
print(f"Time taken for sampling: {end_time - start_time:.4f} seconds")

# now optimization:
start_time = time.time()
w_opt, _ = optimize_model(y_data, x_data, key=SEED, steps=100, step_size=0.0001, init_w=w_random)
end_time = time.time()
time_optimization = end_time - start_time
print(f"Time taken for optimization: {end_time - start_time:.4f} seconds")

ratio = time_optimization / time_sampling
ratio_percent = ratio * 100
print(f"Optimization only takes {ratio_percent:.1f}% of the time of sampling.")
                          

## Multimodality

Even if we initialize well, standard chains (NUTS/HMC) struggle to cross "energy barriers."

In our network, there are two modes: one characterized by the positive $(1, 1)$ and negative $(-1, -1)$ min-norm solution.

### Exercise 2: The Single Long Chain

Let's run a single chain for a long time (5000 samples). Will it explore both modes?

In [ ]:
# Run one very long chain starting again with optimized init

print("Running single long chain (5000 samples)...")
long_chain_samples = run_nuts_sampler(y_data, x_data, w_opt, num_samples=5000, num_warmup=200)

plot_weight_space(long_chain_samples, title="Single Long Chain (5000 samples)", pointalpha=0.01)

**Observation:** The chain is stuck! It explores the top-right mode perfectly but never jumps to the bottom-left mode. The region between them (where $w \approx 0$) has very low probability, acting as a wall.

## The Solution: Hybrid Ensembles

Since a single chain cannot jump modes, we use **ensembling**.

1. Run Optimization multiple times with different random seeds (to find different attraction basins).

2. Launch a *short* NUTS chain from each optimized point.

3. Combine the samples.

### Exercise 3: Multi-Chain Sampling

In [ ]:
# Configuration
NUM_CHAINS = 12
SAMPLES_PER_CHAIN = 250  # 12 * 250 = 3000 Total samples (Less compute budget than above)

ensemble_samples = []

print(f"Running {NUM_CHAINS} chains")
for i in range(NUM_CHAINS): # effectively parallelizable loop!
    print(f"  Chain {i+1}/{NUM_CHAINS}")
    # 1. Optimize
    w_random = jax.random.normal(jax.random.PRNGKey(i), (2,)) * 0.01
    w_opt, _ = optimize_model(y_data, x_data, key=i, steps=100, step_size=0.0001, init_w=w_random)

    # 2. Short(er) Sampling Run
    chain_samples = run_nuts_sampler(
        y_data, x_data, w_opt,
        num_samples=SAMPLES_PER_CHAIN,
        num_warmup=50,
        seed=i
    )
    ensemble_samples.append(chain_samples)

# 3. Combine
combined_samples = jnp.concatenate(ensemble_samples)

plot_weight_space(
    combined_samples, 
    title=f"Ensemble: {NUM_CHAINS} Chains x {SAMPLES_PER_CHAIN} Samples", 
    pointalpha=0.05
)

**Conclusion:** By using optimization to efficiently navigate into high-probability regions, and running many independent chains, we recover the full structure of the posterior (both modes) without specific samplers that can tunnel through barriers if configured correctly.